# Fresh-computer setup

For a completely fresh Windows setup, first double-click **`INSTALL_EVERYTHING_AND_OPEN_NOTEBOOK.bat`** in the project folder. It creates `.venv`, installs all libraries, downloads the public Kaggle/Zenodo datasets, rebuilds the master dataset, downloads the pretrained DistilBERT base weights, registers the Jupyter kernel, and opens this notebook.

For a fully automated classical training run as well, use **`ONE_CLICK_FULL_SETUP_AND_TRAIN.bat`**.


# CS0617 — Complete 10-Week AI Phishing Project
## Large-data training, EDA, baselines, robustness, error analysis, visuals, and interface

**Project:** Robust Detection of AI-Generated Phishing Emails under Cross-Generator Distribution Shift

This notebook is the main execution and evidence notebook for the project. It is deliberately organized around the **10-week CS0617 roadmap**, not only around model training.

### Implemented research question
> How well do text-based phishing-email detectors trained on a large heterogeneous corpus generalize to phishing produced by an unseen LLM generator?

### Important limitation
The available data contain reliable generator labels for GPT-4.1, DeepSeek 3.2, and LLaMA 3.3 70B, but **do not contain genuine adversarial writing-register labels**. Therefore, the notebook evaluates **cross-generator robustness** and does not claim completion of the originally proposed combined generator + register experiment.


## Roadmap alignment

| Course stage | Evidence produced by this notebook/package |
|---|---|
| Week 1 | Problem, question, scope, contribution, experiment plan |
| Weeks 2–3 | Related-work positioning and empirical gap |
| Week 4 | Data acquisition, statistics, EDA, deduplication, leakage checks |
| Week 5 | Dummy + LR + SVM baselines, fixed split, metrics, config |
| Weeks 6–7 | ID test + leave-one-generator-out robustness + F1 drop |
| Weeks 8–9 | Error cases, subgroups, calibration, interpretation |
| Week 10 | Reproducible notebook, figures, CSVs, interface, report-ready outputs |


# Weeks 2–3 — Literature review and positioning

The course requires 6–10 directly relevant papers, comparison of datasets/tasks/metrics/baselines, and a defensible empirical gap.

The package includes `docs/literature_matrix.csv` with eight directly relevant studies. The table below is used to connect the literature review to the implemented robustness question.


In [ ]:
# Literature matrix used for Weeks 2–3
lit_path = ROOT / "docs" / "literature_matrix.csv"
if lit_path.exists():
    literature = pd.read_csv(lit_path)
    display(literature)
    print("Directly relevant papers in matrix:", len(literature))
else:
    print("Literature matrix not found.")


In [ ]:
# 0) Project engine and switches
from pathlib import Path
import sys, subprocess, os, json, csv, socket, time, webbrowser, warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    candidate = Path(r"D:\Ai projects")
    if candidate.exists():
        ROOT = candidate

VENV_PY = ROOT / ".venv" / "Scripts" / "python.exe"
EXEC_PY = VENV_PY if VENV_PY.exists() else Path(sys.executable)

USE_LARGE_DATASET = True
DOWNLOAD_LARGE_DATASET = True
TRAIN_BASELINES = True
RUN_CROSS_GENERATOR = True
RUN_ERROR_ANALYSIS = True
GENERATE_FIGURES = True
LAUNCH_INTERFACE = True
TRAIN_DISTILBERT = False   # Optional, can be enabled later.

print("Project root:", ROOT)
print("Execution Python:", EXEC_PY)
print("Existing .venv reused:", VENV_PY.exists())
print("Large-data mode:", USE_LARGE_DATASET)

if not (ROOT / "src").exists():
    raise FileNotFoundError(r"Project folder not found. Open this notebook from D:\Ai projects.")


In [ ]:
# 1) Dependency verification — installs only missing core packages
core_imports = {
    "pandas":"pandas",
    "numpy":"numpy",
    "sklearn":"scikit-learn",
    "matplotlib":"matplotlib",
    "joblib":"joblib",
    "yaml":"pyyaml",
    "kagglehub":"kagglehub",
    "streamlit":"streamlit",
}

def import_ok(module):
    r = subprocess.run([str(EXEC_PY), "-c", f"import {module}"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return r.returncode == 0

missing = [pkg for module,pkg in core_imports.items() if not import_ok(module)]
if missing:
    print("Installing missing core packages:", missing)
    subprocess.check_call([str(EXEC_PY), "-m", "pip", "install", "-r", str(ROOT/"requirements_core.txt")])
else:
    print("Core environment is ready.")


# Week 4 — Data acquisition, understanding, and quality

The course requires dataset statistics/visualization, a data dictionary, a fixed preprocessing/splitting pipeline, and explicit leakage checks.  
This section downloads the large public corpus, merges it with the generator-labelled data, removes exact duplicates by SHA-256 hash, and produces the final `master_dataset.csv`.


In [ ]:
# 2) Download the large Kaggle dataset if needed
large_dir = ROOT / "data" / "raw" / "kaggle_large"
large_csvs = list(large_dir.rglob("*.csv")) if large_dir.exists() else []

if USE_LARGE_DATASET and DOWNLOAD_LARGE_DATASET and not large_csvs:
    print("Large dataset not found locally. Starting Kaggle download...")
    result = subprocess.run([str(EXEC_PY), str(ROOT/"src"/"download_large_kaggle.py")], cwd=str(ROOT))
    if result.returncode != 0:
        raise RuntimeError(
            "Large dataset download did not complete. "
            "Use the manual fallback printed above, then rerun this cell."
        )
else:
    print("Large dataset already present or download disabled.")
    print("CSV files found:", len(large_csvs))


In [ ]:
# 3) Build the final master dataset
cmd = [str(EXEC_PY), str(ROOT/"src"/"build_master_dataset.py")]
if not USE_LARGE_DATASET:
    cmd.append("--small-only")
subprocess.check_call(cmd, cwd=str(ROOT))

stats_path = ROOT/"data"/"processed"/"master_dataset_stats.json"
stats = json.loads(stats_path.read_text(encoding="utf-8"))
print(json.dumps(stats, indent=2, ensure_ascii=False))


In [ ]:
# 4) Load the master dataset and display the main statistics
import pandas as pd
import numpy as np
from IPython.display import display, HTML

MASTER = ROOT/"data"/"processed"/"master_dataset.csv"
df = pd.read_csv(MASTER, low_memory=False)

summary = pd.DataFrame({
    "Metric":[
        "Total unique emails",
        "Phishing",
        "Legitimate",
        "LLM-origin",
        "Human-origin",
        "Distinct datasets",
        "Distinct generators"
    ],
    "Value":[
        len(df),
        int((df.phishing_label==1).sum()),
        int((df.phishing_label==0).sum()),
        int((df.origin=="llm").sum()),
        int((df.origin=="human").sum()),
        int(df.dataset.nunique()),
        int(df.generator.nunique()),
    ]
})
display(summary)
print("Columns:", list(df.columns))


In [ ]:
# 5) Week-4 EDA charts — dataset composition
import matplotlib.pyplot as plt

fig_dir = ROOT/"results"/"figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Class distribution
class_counts = df["phishing_label"].map({0:"Legitimate",1:"Phishing"}).value_counts()
ax = class_counts.plot(kind="bar", figsize=(7,4), title="Class distribution")
ax.set_xlabel("Class"); ax.set_ylabel("Emails")
plt.tight_layout(); plt.show()

# Dataset/source distribution
ax = df["dataset"].fillna("unknown").value_counts().head(12).sort_values().plot(
    kind="barh", figsize=(8,5), title="Dataset-source distribution"
)
ax.set_xlabel("Emails"); ax.set_ylabel("Dataset")
plt.tight_layout(); plt.show()

# Origin distribution
ax = df["origin"].fillna("unknown").value_counts().head(10).plot(
    kind="bar", figsize=(7,4), title="Origin distribution"
)
ax.set_xlabel("Origin"); ax.set_ylabel("Emails")
plt.tight_layout(); plt.show()

# Generator distribution
ax = df["generator"].fillna("unknown").value_counts().head(12).sort_values().plot(
    kind="barh", figsize=(8,5), title="Generator distribution"
)
ax.set_xlabel("Emails"); ax.set_ylabel("Generator")
plt.tight_layout(); plt.show()

# Email length
word_len = df["word_length"] if "word_length" in df.columns else df["text"].astype(str).str.split().str.len()
plt.figure(figsize=(8,4))
plt.hist(word_len.clip(upper=word_len.quantile(.99)), bins=50)
plt.title("Email-length distribution (trimmed at 99th percentile)")
plt.xlabel("Words"); plt.ylabel("Emails")
plt.tight_layout(); plt.show()


In [ ]:
# 6) Data-quality and leakage checklist
quality = pd.DataFrame({
    "Check":[
        "Missing text",
        "Empty text",
        "Exact duplicate text hashes",
        "Conflicting labels for same text hash",
        "Missing phishing labels"
    ],
    "Count":[
        int(df["text"].isna().sum()),
        int(df["text"].fillna("").astype(str).str.strip().eq("").sum()),
        int(df["text_hash"].duplicated().sum()) if "text_hash" in df.columns else np.nan,
        int(df.groupby("text_hash")["phishing_label"].nunique().gt(1).sum()) if "text_hash" in df.columns else np.nan,
        int(df["phishing_label"].isna().sum())
    ]
})
display(quality)

print("Data dictionary:", ROOT/"docs"/"DATA_DICTIONARY.md")
print("Final dataset stats:", stats_path)


# Week 5 — Baselines and experimental setup

The course asks for at least one simple baseline plus 1–2 strong standard baselines, fixed data splits, metrics, seeds, and reproducible configuration.

This project now uses:
- **DummyClassifier** — trivial baseline
- **TF-IDF + Logistic Regression**
- **TF-IDF + Linear SVM**
- fixed **70/15/15** stratified train/validation/test split
- `random_state = 42`


In [ ]:
# 7) Train/evaluate the baselines on the LARGE master dataset
if TRAIN_BASELINES:
    subprocess.check_call(
        [str(EXEC_PY), str(ROOT/"src"/"train_baselines.py"), "--seed", "42"],
        cwd=str(ROOT)
    )

metrics = pd.read_csv(ROOT/"results"/"baseline"/"baseline_metrics.csv")
test_metrics = metrics[metrics["split"].eq("test")].copy()
display(test_metrics)


In [ ]:
# 8) Baseline-performance chart
metric_cols = ["accuracy","balanced_accuracy","precision","recall","f1","roc_auc","mcc"]
plot_df = test_metrics.set_index("model")[metric_cols].T
ax = plot_df.plot(kind="bar", figsize=(11,5), title="Held-out test performance")
ax.set_xlabel("Metric"); ax.set_ylabel("Score")
plt.xticks(rotation=35, ha="right")
plt.tight_layout(); plt.show()


In [ ]:
# 9) Confusion matrices
cms = pd.read_csv(ROOT/"results"/"baseline"/"baseline_confusion_matrices.csv")
cms = cms[cms["split"].eq("test")]

for _, r in cms.iterrows():
    mat = np.array([[r.tn,r.fp],[r.fn,r.tp]], dtype=int)
    plt.figure(figsize=(4.8,4))
    plt.imshow(mat)
    plt.title(f"Confusion matrix — {r.model}")
    plt.xticks([0,1],["Pred Legit","Pred Phish"])
    plt.yticks([0,1],["True Legit","True Phish"])
    for i in range(2):
        for j in range(2):
            plt.text(j,i,str(mat[i,j]),ha="center",va="center")
    plt.tight_layout(); plt.show()


In [ ]:
# 10) ROC and Precision–Recall curves
roc = pd.read_csv(ROOT/"results"/"baseline"/"roc_points.csv")
roc = roc[roc["split"].eq("test")]
plt.figure(figsize=(6,5))
for model,g in roc.groupby("model"):
    plt.plot(g.fpr,g.tpr,label=model)
plt.plot([0,1],[0,1],linestyle="--")
plt.title("ROC curves — held-out test")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.legend(); plt.tight_layout(); plt.show()

pr = pd.read_csv(ROOT/"results"/"baseline"/"pr_points.csv")
pr = pr[pr["split"].eq("test")]
plt.figure(figsize=(6,5))
for model,g in pr.groupby("model"):
    plt.plot(g.recall,g.precision,label=model)
plt.title("Precision–Recall curves — held-out test")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 11) Interpretable lexical evidence — top weighted terms
terms = pd.read_csv(ROOT/"results"/"baseline"/"top_weighted_terms.csv")
for model in ["logreg","linear_svm"]:
    ph = terms[(terms.model.eq(model)) & (terms.direction.eq("phishing"))].head(15)
    if len(ph):
        plt.figure(figsize=(8,5))
        plt.barh(ph.term[::-1], ph.weight[::-1])
        plt.title(f"Top phishing-associated terms — {model}")
        plt.xlabel("Model weight")
        plt.tight_layout(); plt.show()


# Weeks 6–7 — Core experiments

The main robustness experiment is **Leave-One-Generator-Out**:

1. hold out one LLM generator from training;
2. train on the remaining phishing sources + legitimate data;
3. test on phishing from the unseen generator + held-out legitimate data;
4. repeat for GPT-4.1, DeepSeek 3.2, and LLaMA 3.3 70B;
5. run the experiment for both Logistic Regression and Linear SVM;
6. compare OOD F1 against in-distribution test F1.


In [ ]:
# 12) Cross-generator robustness
if RUN_CROSS_GENERATOR:
    subprocess.check_call(
        [str(EXEC_PY), str(ROOT/"src"/"evaluate_cross_generator.py")],
        cwd=str(ROOT)
    )

cg = pd.read_csv(ROOT/"results"/"baseline"/"cross_generator_phishing_detection.csv")
display(cg)


In [ ]:
# 13) Cross-generator F1 and Relative F1 Drop charts
pivot = cg.pivot(index="held_out_generator",columns="model",values="f1")
ax = pivot.plot(kind="bar", figsize=(9,5), title="Cross-generator F1")
ax.set_xlabel("Held-out generator"); ax.set_ylabel("F1"); ax.set_ylim(0,1.03)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

pivot2 = cg.pivot(index="held_out_generator",columns="model",values="relative_f1_drop_pct")
ax = pivot2.plot(kind="bar", figsize=(9,5), title="Relative F1 change vs. ID test")
ax.set_xlabel("Held-out generator"); ax.set_ylabel("Relative F1 drop (%)")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


# Weeks 8–9 — Error analysis and interpretation

The course explicitly asks for systematic failure cases, subgroup analysis, uncertainty/calibration when relevant, robustness assumptions, and threats to validity.

This section produces:
- false positives and false negatives;
- performance by origin, dataset, generator, and email length;
- Logistic Regression calibration;
- error-count visualization.


In [ ]:
# 14) Run error and subgroup analysis
if RUN_ERROR_ANALYSIS:
    subprocess.check_call([str(EXEC_PY), str(ROOT/"src"/"error_analysis.py")], cwd=str(ROOT))

subgroups = pd.read_csv(ROOT/"results"/"analysis"/"subgroup_metrics.csv")
errors = pd.read_csv(ROOT/"results"/"analysis"/"error_examples.csv")
err_counts = pd.read_csv(ROOT/"results"/"analysis"/"error_counts.csv")

print("Number of held-out test errors:", len(errors))
display(err_counts)
display(subgroups.sort_values(["model","subgroup","f1"]).head(30))


In [ ]:
# 15) Error-count chart and qualitative examples
if not err_counts.empty:
    ax = err_counts.pivot(index="model",columns="error_type",values="count").fillna(0).plot(
        kind="bar", figsize=(8,4), title="False-positive and false-negative counts"
    )
    ax.set_xlabel("Model"); ax.set_ylabel("Count")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()

cols = [c for c in ["model","error_type","score","origin","generator","subject","text"] if c in errors.columns]
display(errors[cols].head(12))


In [ ]:
# 16) Logistic Regression calibration
cal_path = ROOT/"results"/"analysis"/"logreg_calibration.csv"
if cal_path.exists():
    cal = pd.read_csv(cal_path)
    cal = cal[cal["n"] > 0]
    display(cal)
    plt.figure(figsize=(6,5))
    plt.plot(cal.mean_predicted_probability, cal.observed_phishing_rate, marker="o")
    plt.plot([0,1],[0,1],linestyle="--")
    plt.title("Logistic Regression calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed phishing rate")
    plt.tight_layout(); plt.show()


# Week 10 — Reproducibility and final delivery

The notebook now exports report-ready figures and tables, preserves fixed split IDs, saves trained models, and can launch the interactive Streamlit interface.


In [ ]:
# 17) Generate and export report-ready figures/tables
if GENERATE_FIGURES:
    subprocess.check_call([str(EXEC_PY), str(ROOT/"src"/"generate_figures.py")], cwd=str(ROOT))
    subprocess.check_call([str(EXEC_PY), str(ROOT/"src"/"export_visual_ready.py")], cwd=str(ROOT))

print("Figures:", ROOT/"results"/"figures")
print("Visual-ready tables:", ROOT/"results"/"visual_ready")


In [ ]:
# 18) Display all generated figures inside the notebook
from IPython.display import Image, display
for p in sorted((ROOT/"results"/"figures").glob("*.png")):
    print("\n", p.name)
    display(Image(filename=str(p), width=850))


## Optional Transformer extension — DistilBERT

DistilBERT remains optional because it can take substantially longer on CPU.  
If your final claim compares **classical ML vs transformers**, then you should enable this stage and report the DistilBERT results on the same fixed data split.


In [ ]:
# 19) OPTIONAL DistilBERT
if TRAIN_DISTILBERT:
    optional_imports = ["torch","transformers","datasets","accelerate","scipy"]
    missing_optional = [m for m in optional_imports if not import_ok(m)]
    if missing_optional:
        subprocess.check_call(
            [str(EXEC_PY), "-m", "pip", "install", "-r", str(ROOT/"requirements_transformer.txt")]
        )
    subprocess.check_call([
        str(EXEC_PY), str(ROOT/"src"/"train_distilbert.py"),
        "--epochs","2","--batch-size","8","--max-length","256"
    ], cwd=str(ROOT))
    print("DistilBERT saved to", ROOT/"results"/"distilbert_model")
else:
    print("DistilBERT skipped. Set TRAIN_DISTILBERT=True only when you are ready.")


In [ ]:
# 20) Launch the Streamlit interface
def port_open(host="127.0.0.1", port=8501):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(.4)
        return s.connect_ex((host,port)) == 0

url = "http://localhost:8501"
if LAUNCH_INTERFACE:
    if not port_open():
        log_path = ROOT/"results"/"streamlit.log"
        log_file = open(log_path,"a",encoding="utf-8")
        proc = subprocess.Popen(
            [str(EXEC_PY),"-m","streamlit","run",str(ROOT/"app.py"),
             "--server.headless=true","--server.port=8501"],
            cwd=str(ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            creationflags=getattr(subprocess,"CREATE_NEW_PROCESS_GROUP",0),
        )
        (ROOT/"results"/"streamlit.pid").write_text(str(proc.pid),encoding="utf-8")
        time.sleep(4)
    print("Interface:", url)
    try:
        webbrowser.open(url)
    except Exception:
        pass
else:
    print("Interface launch disabled.")


# Final scientific checklist

Before submitting, verify that you can answer all of the following:

1. What exactly is the research question?
2. Which hypotheses or expected outcomes were tested?
3. What is the final post-deduplication dataset size?
4. Why is DummyClassifier included?
5. Why are Logistic Regression and Linear SVM appropriate baselines?
6. What is the difference between ID testing and cross-generator OOD testing?
7. Which generator is hardest to generalize to?
8. How much does F1 change under generator shift?
9. Where does the system fail (FP/FN and subgroups)?
10. What can the evidence **not** establish?
11. Can another student reproduce the result from this notebook?

**Do not claim generator + adversarial-register robustness unless register-labelled data are added.**
